# SAIGE — behavioral eval (free Colab T4 + Claude API)

**The question.** The corrected logprob ablation found run3 moves every held-out
scenario positive, p = 0.0039 clustered, surviving per-token normalization — but
with **zero preference flips**. Not one record changed which response the model
prefers. So: does +5.33 nats of margin correspond to any difference in what the
model actually *says*?

**The method.** Generate from base / run3 / run4 / run5 across all three prompt
conditions on the same 11 held-out records the logprob ablation scored, then
score every generation with `score_candidate` from `generate_dpo_pairs.py` — the
same anchored Claude judge whose calibration the `chosen_score` / `rejected_score`
values in `dpo_pairs.jsonl` share.

**The headline is the `generic` column.** Per `local-trainer/README.md`: *"If
adapter + generic looks close to adapter + RS, the training worked"* and
*"`SAIGE+gen - base+gen` (behavior in the weights, no RS prompt) is the headline
number."*

**Blinding.** The judge sees only (user message, response, annotation record) —
never which model or condition produced a response.

**Cost.** GPU generation is free. Judging is **132 Claude API calls**, cached, so
re-runs only pay for calls that have not succeeded yet.

**Runtime.** Roughly **15–25 min** end to end: a few minutes of first-run model
download, generation, then judging. These are estimates, not T4 measurements.
Every long step prints progress as it goes, so a quiet cell means something is
wrong.

**Keeping the session alive.** Keep this tab in the foreground. If Colab shows a
CAPTCHA or asks whether you are still there, answer it — an unanswered check ends
the session. VPNs and ad blockers make both disconnects and CAPTCHAs more likely.
Progress is checkpointed, so if the runtime does drop, re-run from section 4 and
it resumes.

## 1. Confirm a GPU

In [ ]:
import subprocess
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception:
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then rerun.")
import torch
assert torch.cuda.is_available()
p = torch.cuda.get_device_properties(0)
print(f"{p.name} | {p.total_memory/1e9:.1f} GB | cc {torch.cuda.get_device_capability(0)}")

## 2. Dependencies

In [ ]:
%pip install -q -U "transformers>=4.45" "datasets>=2.20" "peft>=0.11" \
    "bitsandbytes>=0.43" "accelerate>=0.30" huggingface_hub anthropic

import transformers, datasets, peft, anthropic, torch
for m in (torch, transformers, datasets, peft, anthropic):
    print(f"{m.__name__:16s} {m.__version__}")

## 3. Hugging Face authentication

Everything this notebook reads is public, so a token is optional — but anonymous
downloads are rate-limited and slower, which is felt most on mobile while pulling
the ~2 GB base model and three adapters.

**Set it once instead of typing it every run:** tap the key icon in Colab's left
sidebar, add a secret named `HF_TOKEN`, and toggle notebook access on. This cell
picks it up automatically from then on. Otherwise it prompts, with the input
hidden.

A **read** token is enough. Only use a write token if you intend to push results
back to the Hub from here.

In [ ]:
import os, getpass
from huggingface_hub import login, whoami

tok = os.environ.get("HF_TOKEN", "")

# Colab Secrets first — the mobile-friendly path, no typing on later runs.
if not tok:
    try:
        from google.colab import userdata
        tok = (userdata.get("HF_TOKEN") or "").strip()
        if tok:
            print("token loaded from Colab Secrets")
    except Exception:
        pass   # secret absent, or access not granted to this notebook

if not tok:
    print("No HF_TOKEN found in the environment or Colab Secrets.")
    print("Paste a token (starts with hf_), or press Enter to continue anonymously.")
    tok = getpass.getpass("HF token (input hidden): ").strip()

if tok:
    # Both names are set because the later cells run eval_behavior.py as a
    # SUBPROCESS, which inherits os.environ but not this kernel's session state.
    os.environ["HF_TOKEN"] = tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
    try:
        login(token=tok, add_to_git_credential=False)
        print("authenticated as:", whoami().get("name"))
    except Exception as e:
        print(f"WARNING: token rejected ({type(e).__name__}: {e})")
        print("  Continuing anonymously — the public repos still work.")
        os.environ.pop("HF_TOKEN", None)
        os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)
else:
    print("Continuing WITHOUT a token.")
    print("  Public repos still work, but expect slower, rate-limited downloads.")

## 4. Get the project (the script is embedded in this notebook)

Clones the SAIGE repo for the judge and the annotation records, then writes out
`eval_behavior.py` from a copy embedded here — overwriting whatever is on disk,
so the notebook and the script can never disagree. Nothing to upload.

**Re-run this cell after any reconnect**: it re-creates `LT`, `WORK` and
`run_step`, which live only in the kernel's memory.

In [ ]:
import os, subprocess, sys, base64, hashlib
from pathlib import Path

# Absolute anchor, never cwd: a runtime restart resets cwd to /content.
ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
if not (ROOT/"SAIGE").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/M1ztick/SAIGE.git"], cwd=ROOT, check=True)
LT = (ROOT/"SAIGE"/"local-trainer").resolve()
assert (LT/"generate_dpo_pairs.py").exists(), f"repo layout unexpected at {LT}"
assert (LT/"annotations").is_dir(), f"annotations/ missing at {LT}"
os.chdir(LT)
print("project:", LT)

EVAL_BEHAVIOR_B64 = "..."
EVAL_BEHAVIOR_SHA = "d86df0d6df43"
blob = base64.b64decode(EVAL_BEHAVIOR_B64)
assert hashlib.sha256(blob).hexdigest()[:12] == EVAL_BEHAVIOR_SHA, "embedded script corrupted"
(LT/"eval_behavior.py").write_bytes(blob)
print(f"wrote eval_behavior.py ({len(blob)} bytes, sha {EVAL_BEHAVIOR_SHA})")

p = LT/"ablation_v3.py"
if p.exists():
    stale = "scenario" not in p.read_text()
    print(f"ablation_v3.py present ({'STALE - pre-clustering' if stale else 'current'})"
          + (" — cross-check will be skipped; don't use it for the logprob re-run" if stale else ""))

# Where generations, the checkpoint, the judge cache and results go. Section 5
# can switch this to Google Drive.
WORK = globals().get("WORK", LT)
print("work dir:", WORK)

def run_step(*args, env_extra=None):
    """Run one phase with LIVE output, and STOP if it fails.

    The previous version captured output and showed it only when the step
    ended, so a working hour-long step and a hung one looked identical. This
    prints every line as it happens and still raises on failure.
    """
    env = {**os.environ, "PYTHONUNBUFFERED": "1",
           "HF_HUB_DISABLE_PROGRESS_BARS": "1",   # \r bars would flood the cell
           **(env_extra or {})}
    cmd = [sys.executable, "-u", "eval_behavior.py", *args, "--work-dir", str(WORK)]
    print("$ python", " ".join(cmd[2:]), "\n", flush=True)
    with subprocess.Popen(cmd, cwd=LT, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as pr:
        for line in pr.stdout:
            print(line, end="", flush=True)
        rc = pr.wait()
    if rc != 0:
        raise RuntimeError(f"eval_behavior.py {' '.join(args)} exited {rc} — "
                           f"the real error is in the output just above.")

print("run_step() ready")

## 5. (Optional) Keep progress on Google Drive

Checkpoints and the judge cache normally live in the runtime's own disk. That
survives a dropped connection or a kernel restart, but **not** Colab recycling
the runtime — which wipes the disk. Set `PERSIST_TO_DRIVE = True` to keep them on
your Drive instead; mounting asks you to sign in once.

Skip this cell if you'd rather not — everything still works.

In [ ]:
PERSIST_TO_DRIVE = False

if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/saige_behavior")
else:
    WORK = LT
WORK.mkdir(parents=True, exist_ok=True)
print("work dir:", WORK)

## 6. Dry run — no GPU, no API

Checks that the `rs` condition is the prompt the adapters were **trained** under
(`diversify_prompts.RS_PROMPT`), that every eval prompt resolves to an annotation
record, and that the statistics work. Under a minute. **If this fails, stop.**

In [ ]:
run_step("--repo", ".", env_extra={"SAIGE_DRY_RUN": "1"})

## 7. Generate (GPU, free)

One batched pass per model — all 33 prompts at once — so the four passes should
take minutes rather than the hours the one-at-a-time version needed. The first
run also downloads the ~2 GB base model and three adapters.

You'll see a line per pass as it finishes. **Each finished pass is checkpointed:**
if the runtime drops, re-run sections 4 and 7 and it picks up where it stopped.
Watch the `truncated` count — a response cut off at the token cap gets penalised
by the judge.

In [ ]:
run_step("--repo", ".", "--phase", "generate")

import json
raw = json.loads((WORK/"behavior_generations.json").read_text())
g = raw["generations"] if isinstance(raw, dict) else raw
print(f"\nOK: {len(g)} generations in {WORK}")
for st in dict.fromkeys(x["state"] for x in g):
    n = sum(x["truncated"] for x in g if x["state"] == st)
    print(f"  {st:5s} truncated {n:2d}/{sum(x['state']==st for x in g)}")

## 8. Judge (Claude API, no GPU)

132 calls, six at a time, so a few minutes. Failed calls are retried. Any that
still fail are **excluded from the analysis — never counted as a score** — and
re-running this step retries only those, since everything that succeeded is
cached.

Paste an API key below, or add `ANTHROPIC_API_KEY` to Colab Secrets to skip the
prompt next time.

In [ ]:
import os, getpass

key = os.environ.get("ANTHROPIC_API_KEY", "")

# Same Colab Secrets pattern — add ANTHROPIC_API_KEY there to skip this prompt.
if not key:
    try:
        from google.colab import userdata
        key = (userdata.get("ANTHROPIC_API_KEY") or "").strip()
        if key:
            print("key loaded from Colab Secrets")
    except Exception:
        pass

if not key:
    key = getpass.getpass("ANTHROPIC_API_KEY (input hidden): ").strip()

assert key, "the judge phase needs an API key"
os.environ["ANTHROPIC_API_KEY"] = key   # inherited by the subprocess below
print(f"key set ({len(key)} chars)")

In [ ]:
run_step("--repo", ".", "--phase", "judge")
assert (WORK/"behavior_eval_results.json").exists(), "judging produced no results file"
print(f"\nOK: results at {WORK/'behavior_eval_results.json'}")

## 9. How to read it

**Data quality first.** The table at the top counts, per model, how many responses
were cut off at the token cap and how many could not be judged. A model with many
of either is being compared on a different footing — say so before reading its
numbers.

**Headline** — the `generic` rows, marked `<<`: `adapter+gen - base+gen`, behavior
in the weights with no Right Speech prompt active. Positive and significant there
is the project's own definition of the training working.

**`unconditionality_ratio`** — the generic gain over the rs gain. Near 1.0 means
the adapter behaves the same with or without the RS prompt. The logprob run put
run3 at 0.85 on margins.

**Ties.** Judge scores are integers 0–10, so many paired deltas are exactly 0.
Ties are excluded from the sign test, so a high `tied` count means the test is
underpowered, not that the effect is absent. Read the CI alongside the p-value.

**The comparison that matters** is against the logprob result. If run3 is positive
here too, the margin shift is behavioral. If it is flat here while margins moved,
run3 changed what the model *prefers* without changing what it *emits* — which,
given zero preference flips, is what I'd expect going in.

In [ ]:
import json
from pathlib import Path

W = globals().get("WORK")
cand = [p for p in ([W/"behavior_eval_results.json"] if W else []) if p.exists()]
if not cand:
    for base in (Path("/content/drive/MyDrive/saige_behavior"), Path("/content")):
        if base.exists():
            cand += list(base.rglob("behavior_eval_results.json"))
if not cand:
    raise SystemExit("behavior_eval_results.json does not exist yet — it is written only "
                     "by the judge step. Re-run sections 4 and 8 and read their output.")
path = cand[0]
print(f"reading {path}\n")
r = json.loads(path.read_text())
res = r["results"]
conds = ["rs", "generic", "none"]

dq = res.get("data_quality", {})
if dq:
    print("DATA QUALITY")
    for st, d in dq.items():
        flag = "  <- check" if (d["truncated"] or d["unjudged_excluded"]) else ""
        print(f"  {st:5s} {d['n']:3d} responses | truncated {d['truncated']:3d} "
              f"| unjudged (excluded) {d['unjudged_excluded']:3d}{flag}")

print("\nMEAN JUDGE OVERALL (0-10)")
print(f"{'state':6s}" + "".join(f"{c:>11s}" for c in conds))
for st, a in res["absolute"].items():
    print(f"{st:6s}" + "".join(f"{a['mean_by_condition'].get(c, float('nan')):>11.3f}" for c in conds))

print("\nPAIRED vs base (scenario-clustered) — generic is the headline")
hdr = f"{'state':6s} {'cond':8s} {'mean d':>8s} {'pos/n':>8s} {'tied':>5s} {'p':>8s} {'CI95':>18s}"
print(hdr); print("-" * len(hdr))
for st, cs in res["paired_vs_base_scenario_clustered"].items():
    for c in conds:
        s = cs.get(c)
        if not s:
            continue
        lo, hi = s["bootstrap_ci95"]
        star = " *" if (s["sign_test_p"] or 1) < 0.05 else "  "
        mark = " <<" if c == "generic" else ""
        print(f"{st:6s} {c:8s} {s['mean_delta']:>+8.3f} {s['n_positive']:>3d}/{s['n_nonzero']:<4d} "
              f"{s['n_tied']:>5d} {s['sign_test_p']:>8.4f}{star}[{lo:>+6.2f},{hi:>+6.2f}]{mark}")

print("\nPROJECT COMPARISONS")
for st, cmp_ in res["project_comparisons"].items():
    print(f"  {st}:")
    for k, v in cmp_.items():
        print(f"    {k:<52s} {v:+.3f}" if v is not None else f"    {k:<52s}   n/a")
print("\n* = sign test p < 0.05   << = headline (behavior in the weights)")

## 10. Save

In [ ]:
from google.colab import files
W = globals().get("WORK", Path("."))
for name in ("behavior_eval_results.json", "behavior_generations.json"):
    p = W/name
    if p.exists():
        files.download(str(p))
    else:
        print(f"skipping {name} — not found in {W}")